# Lekcija 09 - Uzorak dizajna metakognicije


## Postavljanje

Ovaj bilježnik demonstrira dizajnerski obrazac Metakognicija koristeći Microsoft Agent Framework.

**Preduvjeti:**
- Azure OpenAI implementacija konfigurirana putem varijabli okruženja
- Azure CLI prijavljen (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Što je metakognicija?

Metakognicija je **razmišljanje o razmišljanju**. U kontekstu AI agenata, to znači izgradnju agenata koji mogu:

- **Samo-refleksija** o vlastitim rezultatima i procesu zaključivanja
- **Otkrivati pogreške** i oporavljati se na elegantan način umjesto da tiho ne uspiju
- **Procijeniti** jesu li njihovi odgovori potpuni i korisni
- **Prilagoditi** svoju strategiju kad početni pristup ne uspije (npr. prelazak na rezervni sustav)

Agent sa metakognicijom ne samo da odgovara na pitanja — on prati vlastitu izvedbu i prilagođava se u hodu.


## Primarni i rezervni alati

Uobičajeni metakognitivni obrazac je **strategija povratka**. Agent prvo pokušava s primarnim alatom; ako ne uspije (npr. pogreška 404), agent prepoznaje neuspjeh i transparentno prelazi na rezervni alat.

Ovo odražava stvarne sustave u kojima primarne usluge mogu biti nedostupne i agenti moraju samostalno dijagnosticirati problem prije nego što odaberu alternativni put.

U nastavku definiramo dva alata za pretragu leta:
- **Primarni** — pokriva Pariz, Tokio i Barcelonu
- **Rezervni** — pokriva Berlin, Sydney i New York


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent sa samorefleksijom i oporavkom od pogrešaka

Agent ispod je uputstvom da prvo pokuša glavni sustav leta, prepozna pogreške i transparentno se prebaci na rezervni sustav. Nakon svakog odgovora, kratko se samoprocjenjuje je li u potpunosti odgovorio na korisnikovo pitanje.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Uzorak samoprocjene

Još jedan aspekt metakognicije je **samoprocjena**: poseban agent (ili isti agent u drugom prolazu) pregledava odgovor radi potpunosti, točnosti i korisnosti.

U nastavku stvaramo agenta `ResponseEvaluator` koji ocjenjuje odgovore turističkih agenata u tri dimenzije.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sažetak

U ovoj lekciji ste naučili kako izgraditi **metakognitivne agente** koristeći Microsoft Agent Framework:

- **Samorefleksija**: Agenti koji prate vlastito razmišljanje i transparentno komuniciraju što se dogodilo.
- **Oporavak od pogrešaka s rezervnim opcijama**: Uzorak primarnog + rezervnog alata gdje agent otkriva neuspjehe (npr. 404 pogreške) i automatski pokušava alternativni izvor.
- **Samoocjena**: Poseban evaluacijski agent koji ocjenjuje odgovore na potpunost, točnost i korisnost.

Ovi obrasci čine agente robusnijima, transparentnijima i pouzdanijima — što su ključne kvalitete za proizvodna okruženja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
